In [30]:
#load libraries
require(tidyverse)
require(dplyr)

In [41]:
#load Colony_Data.csv
colony=read.csv("~/Documents/SCTLD/SCTLD_samples/Sample_Data/CBC_ColonyData.csv")

In [42]:
colnames(colony)

[1] "X"                        "Date_InitialTag"         
 [3] "Transect"                 "TransectNum"             
 [5] "OldTagNum"                "NewTagNum"               
 [7] "Species"                  "Meter"                   
 [9] "Meters_90"                "Direction"               
[11] "Size_Class"               "MaxDiameter"             
[13] "Height"                   "Date_DocumentedDisease"  
[15] "Date_DocumentedMortality" "Notes_062019"            
[17] "X062019_Condition"        "X062019_Percentage"      
[19] "X102019_Condition"        "X102019_Percentage"      
[21] "Notes_052022"             "X052022_Condition"       
[23] "X052022_Percentage"       "Notes_122022"            
[25] "X122022_Condition"        "X122022_Percentage"      
[27] "Notes_092023"             "X092023_Condition"       
[29] "X092023_Percentage"       "Notes_112023"            
[31] "X112023_Condition"        "X112023_Percentage"      
[33] "Notes_122023"             "X122023_Condition"       
[35] "X122023_Percentage"       "Notes_012024"            
[37] "X012024_Condition"        "X012024_Percentage"      
[39] "Notes_022024"             "X022024_Condition"       
[41] "X022024_Percentage"       "Notes_042024"            
[43] "X042024_Condition"        "X042024_Percentage"      
[45] "X062024_Condition"        "X062024_Percentage"      
[47] "Notes_062024"             "X082024_Condition"       
[49] "X082024_Percentage"       "Notes_082024"            
[51] "X122024_Condition"        "X122024_Percentage"      
[53] "Notes_122024"             "X062025_Condition"       
[55] "X062025_Percentage"       "Notes_062025"            
[57] "immune_y.n"               "checked_colonies"

In [43]:
#cleaning it up to only include conditions, transect num and tag num
#cclean to just include Condition columns and colony'
colony=colony[c('TransectNum', 'NewTagNum', 'Date_DocumentedDisease', 'X062019_Condition', 'X102019_Condition', 'X052022_Condition', 'X122022_Condition', 'X092023_Condition', 'X112023_Condition', 'X122023_Condition','X012024_Condition','X022024_Condition', 'X042024_Condition', 'X062024_Condition', 'X082024_Condition', 'immune_y.n')]
head(colony)

,TransectNum,NewTagNum,Date_DocumentedDisease,X062019_Condition,X102019_Condition,X052022_Condition,X122022_Condition,X092023_Condition,X112023_Condition,X122023_Condition,X012024_Condition,X022024_Condition,X042024_Condition,X062024_Condition,X082024_Condition,immune_y.n
,<int>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
1,1,1,5/21/22,Healthy,,Diseased,Diseased,Diseased,Not_Visited,Not_Visited,Diseased,Not_Visited,Diseased,Not_Visited,Not_Visited,n
2,1,2,Healthy,Healthy,,Healthy,Healthy,"CLP,CLB",CLB,CLB,"CLP,CLB",Healthy,Not_Visited,Healthy,Healthy,y
3,1,3,5/21/22,Healthy,,Diseased,Healthy,Diseased,CLP,Healthy,Healthy,Diseased,Diseased,Diseased,Diseased,y
4,1,4,5/21/22,Healthy,Healthy,Diseased,Dead,Not_Visited,Not_Visited,Not_Visited,Not_Visited,Not_Visited,Not_Visited,Not_Visited,Not_Visited,n
5,1,5,5/21/22,Healthy,,Diseased,Diseased,Diseased,Not_Visited,Not_Visited,Diseased,Not_Visited,Diseased,Not_Visited,Not_Visited,n
6,1,6,12/2/22,Healthy,Healthy,Healthy,Diseased,Dead,Not_Visited,Not_Visited,Dead,Not_Visited,Not_Visited,Not_Visited,Not_Visited,n


In [44]:
#isolate non-immune samples from colony data and focus on samples that got disease
colony_nonimmune=colony[colony$immune_y.n=="n",]
#i want colonies that got disease, not those that are currently healthy
colony_nonimmune <- colony_nonimmune[colony_nonimmune$Date_DocumentedDisease!= "Healthy",]
#but I also don't want colonies that died before 052022
# colony_nonimmune <- colony_nonimmune[colony_nonimmune$X052022_Condition!= "Dead",]

In [45]:
#make new column colony in colony data to match meta
colony_nonimmune$colony <- c(paste0(colony_nonimmune$TransectNum, "_", colony_nonimmune$NewTagNum))

In [46]:
#rename condition columns to corresponding Monthyear
colony_clean <- colony_nonimmune %>%
  rename_with(
    .cols = matches("^X\\d{6}_Condition$"),
    .fn = ~ str_extract(., "\\d{6}")
  )

#pivot longer colony to join all date columns into one monthyear column
colony_long <- colony_clean %>%
  pivot_longer(
    cols = matches("^\\d{6}$"),      # after renaming
    names_to = "Month_year",
    values_to = "Condition"
  )
head(colony_long)

#remove now unnecessary transect num and tag num
colony_long[ ,c('TransectNum', 'NewTagNum')] <- list(NULL)
head(colony_long)

TransectNum,NewTagNum,Date_DocumentedDisease,immune_y.n,colony,Month_year,Condition
<int>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
1,1,5/21/22,n,1_1,062019,Healthy
1,1,5/21/22,n,1_1,102019,
1,1,5/21/22,n,1_1,052022,Diseased
1,1,5/21/22,n,1_1,122022,Diseased
1,1,5/21/22,n,1_1,092023,Diseased
1,1,5/21/22,n,1_1,112023,Not_Visited


Date_DocumentedDisease,immune_y.n,colony,Month_year,Condition
<chr>,<chr>,<chr>,<chr>,<chr>
5/21/22,n,1_1,062019,Healthy
5/21/22,n,1_1,102019,
5/21/22,n,1_1,052022,Diseased
5/21/22,n,1_1,122022,Diseased
5/21/22,n,1_1,092023,Diseased
5/21/22,n,1_1,112023,Not_Visited


In [47]:
#load metagenomics data
metagenomics=read.csv("~/Documents/SCTLD/SCTLD_samples/Sample_Data/Metagenomics_Tracker_Belize.csv")

In [48]:
sample=read.csv("~/Documents/SCTLD/SCTLD_samples/Sample_Data/CBC_samples.csv")

# sample_dna add colony column to sample data
sample$colony= colony= c(paste0(sample$TransectNum, "_", sample$NewTagNum))

#leading zeros are gone, add them back in
sample$Month_year <- sprintf("%06d", as.numeric(sample$Month_year))

#clean sample data of leading spaces
sample$Month_year <- trimws(sample$Month_year)
sample$Health_status <- trimws(sample$Health_status)

# only utilize correct species, and EtOH + RNALater samples and exclue 062025 samples
sample_DNA <- sample[(sample$Sample_type == "Core_EtOH" | sample$Sample_type == "Core_RNAlater") & 
    sample$Month_year != "062025",]

head(sample_DNA)
#double check that all 062025 samples have been removed
unique(sample_DNA$Month_year)

,Month_year,Country,Location,CollectionDate,Transect,TransectNum,OldTagNum,NewTagNum,Species,Time_sampled,⋯,Sample_type,SampleNum,Health_status,Sampling_notes,Tubelabel_species,Sample_physical_location,Extraction_physical_location,Date_sequenced,Notes,colony
,<chr>,<chr>,<chr>,<chr>,<chr>,<int>,<chr>,<chr>,<chr>,<chr>,⋯,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
2,092023,BEL,CBC,9/27/23,CBC30N,1,,1,SSID,,⋯,Core_RNAlater,185,Diseased_Margin,only margin sample available,092023_BEL_CBC_T1_185_SSID,UML_NARWHAL_R1_B10,,,,1_1
3,092023,BEL,CBC,9/25/23,CBC30N,1,,2,PAST,,⋯,Core_RNAlater,171,Bleached_Tissue,CLP 90%,092023_BEL_CBC_T1_171_PAST,UML_NARWHAL_R1_B10,UML_NARWHAL_R2_B12,,,1_2
4,092023,BEL,CBC,9/25/23,CBC30N,1,,3,SSID,,⋯,Core_RNAlater,173,Healthy,CLP 80%; DC 20%,092023_BEL_CBC_T1_173_SSID,UML_NARWHAL_R1_B10,UML_NARWHAL_R2_B12,,,1_3
5,092023,BEL,CBC,9/25/23,CBC30N,1,,12,PSTR,,⋯,Core_RNAlater,177,Healthy,No CL,092023_BEL_CBC_T1_177_PSTR,UML_NARWHAL_R1_B10,UML_NARWHAL_R2_B12,,,1_12
6,092023,BEL,CBC,9/25/23,CBC30N,1,,13,PAST,,⋯,Core_RNAlater,175,Healthy,No CL,092023_BEL_CBC_T1_175_PAST,UML_NARWHAL_R1_B10,UML_NARWHAL_R2_B12,,R2_B15 EXTRACTED TWICE,1_13
7,092023,BEL,CBC,9/27/23,CBC30N,1,,5,SSID,,⋯,Core_RNAlater,188,Diseased_Tissue,,092023_BEL_CBC_T1_188_SSID,UML_NARWHAL_R1_B10,UML_NARWHAL_R2_B11,,,1_5


[1] "092023" "122022" "052022" "112023" "042024" "102019" "062019" "122023"
 [9] "012024" "022024" "062024" "082024"

In [49]:
#extract rows where my species of interest are
sample_clean <- sample_DNA[sample_DNA$Species == "MCAV"|sample_DNA$Species == "PSTR"|sample_DNA$Species == "SSID"|sample_DNA$Species == "PAST"|sample_DNA$Species == "OFAV"|sample_DNA$Species == "OANN", ]

#view sample_clean, this is a new table modeled after sample data sheet and utilizing the colony tag num as a column
head(sample_clean)

,Month_year,Country,Location,CollectionDate,Transect,TransectNum,OldTagNum,NewTagNum,Species,Time_sampled,⋯,Sample_type,SampleNum,Health_status,Sampling_notes,Tubelabel_species,Sample_physical_location,Extraction_physical_location,Date_sequenced,Notes,colony
,<chr>,<chr>,<chr>,<chr>,<chr>,<int>,<chr>,<chr>,<chr>,<chr>,⋯,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
2,092023,BEL,CBC,9/27/23,CBC30N,1,,1,SSID,,⋯,Core_RNAlater,185,Diseased_Margin,only margin sample available,092023_BEL_CBC_T1_185_SSID,UML_NARWHAL_R1_B10,,,,1_1
3,092023,BEL,CBC,9/25/23,CBC30N,1,,2,PAST,,⋯,Core_RNAlater,171,Bleached_Tissue,CLP 90%,092023_BEL_CBC_T1_171_PAST,UML_NARWHAL_R1_B10,UML_NARWHAL_R2_B12,,,1_2
4,092023,BEL,CBC,9/25/23,CBC30N,1,,3,SSID,,⋯,Core_RNAlater,173,Healthy,CLP 80%; DC 20%,092023_BEL_CBC_T1_173_SSID,UML_NARWHAL_R1_B10,UML_NARWHAL_R2_B12,,,1_3
5,092023,BEL,CBC,9/25/23,CBC30N,1,,12,PSTR,,⋯,Core_RNAlater,177,Healthy,No CL,092023_BEL_CBC_T1_177_PSTR,UML_NARWHAL_R1_B10,UML_NARWHAL_R2_B12,,,1_12
6,092023,BEL,CBC,9/25/23,CBC30N,1,,13,PAST,,⋯,Core_RNAlater,175,Healthy,No CL,092023_BEL_CBC_T1_175_PAST,UML_NARWHAL_R1_B10,UML_NARWHAL_R2_B12,,R2_B15 EXTRACTED TWICE,1_13
7,092023,BEL,CBC,9/27/23,CBC30N,1,,5,SSID,,⋯,Core_RNAlater,188,Diseased_Tissue,,092023_BEL_CBC_T1_188_SSID,UML_NARWHAL_R1_B10,UML_NARWHAL_R2_B11,,,1_5


In [50]:
nrow(sample_DNA)
nrow(sample_clean)

[1] 868

[1] 821

In [51]:
#changing X column to Tubelabel_species
names(metagenomics)[names(metagenomics) == 'X'] <- 'Tubelabel_species'
colnames(metagenomics)

[1] "Tubelabel_species"                                                  
 [2] "Health_Status"                                                      
 [3] "Starting_Weight"                                                    
 [4] "Date_Extracted"                                                     
 [5] "Raw_ng_ul"                                                          
 [6] "Date_Enriched"                                                      
 [7] "Microbe_ng_ul"                                                      
 [8] "Microbe_Location"                                                   
 [9] "Microbe_clean_date.n"                                               
[10] "Host_ng_ul"                                                         
[11] "Host_Location"                                                      
[12] "Host_clean_y.n"                                                     
[13] "Date_Libprep"                                                       
[14] "Status"                                                             
[15] "Notes"                                                              
[16] "Seq_date"                                                           
[17] "Host_Seq_date"                                                      
[18] "Microbe_seq_file"                                                   
[19] "Host_seq_file"                                                      
[20] "Seq_Location..in.Unity."                                            
[21] "Sample_physical_location"                                           
[22] "Extraction_physical_location"                                       
[23] "Location_notes"                                                     
[24] "Sample_Code...datecollected_tagnumber_transect_samplenumber_species"

In [52]:
#the columns that I want to isolate from metagenomics tracker
metagenomics_PCR = metagenomics[c("Tubelabel_species", "Date_Extracted", "Raw_ng_ul", "Date_Enriched","Microbe_Location")]

In [53]:
head(metagenomics_PCR)

,Tubelabel_species,Date_Extracted,Raw_ng_ul,Date_Enriched,Microbe_Location
,<chr>,<chr>,<chr>,<chr>,<chr>
1,052022_BEL_CBC_T2_45_PSTR,10_10_2023,22.8,10_17_2023,UML_NARWHAL_R2_B1
2,052022_BEL_CBC_T2_46_PSTR,10_10_2023,63.9,10_17_2023,UML_NARWHAL_R2_B1
3,052022_BEL_CBC_T2_59_OFAV,10_3_2023,73.7,10_16_2023,UML_NARWHAL_R2_B1
4,042024_BEL_CBC_T1_925_PAST,8_19_2024,5.46,9_28_2024,UML_NARWHAL_R6_B30
5,052022_BEL_CBC_T2_72_OFAV,10_3_2023,47.6,10_11_23,UML_NARWHAL_R6_B1
6,052022_BEL_CBC_T2_71_OFAV,10_3_2023,98.8,10_16_2023,UML_NARWHAL_R6_B1


In [54]:
## now combine sample and colony data by month_year and colony
#merge new colony_long and meta
meta<- merge(sample_clean, colony_long,
                     by = c("Month_year", "colony"),
                     all = FALSE)

In [55]:
head(meta)
nrow(meta)

,Month_year,colony,Country,Location,CollectionDate,Transect,TransectNum,OldTagNum,NewTagNum,Species,⋯,Health_status,Sampling_notes,Tubelabel_species,Sample_physical_location,Extraction_physical_location,Date_sequenced,Notes,Date_DocumentedDisease,immune_y.n,Condition
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<int>,<chr>,<chr>,<chr>,⋯,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
1,042024,1_1,BEL,CBC,4/25/24,CBC30N,1,,1,SSID,⋯,Diseased_Margin,99% CTL/ old mort - only margin sample available,042024_BEL_CBC_T1_937_SSID,UML_NARWHAL_R5_B24,,,,5/21/22,n,Diseased
2,042024,1_11,BEL,CBC,4/25/24,CBC30N,1,,11,SSID,⋯,Diseased_Margin,95% CTL/ old mortality - only margin sample available,042024_BEL_CBC_T1_936_SSID,UML_NARWHAL_R5_B24,,,,5/21/22,n,Diseased
3,042024,1_16,BEL,CBC,4/25/24,CBC30N,1,,16,SSID,⋯,Diseased_Margin,"50%TL sub acute, 17% acute",042024_BEL_CBC_T1_939_SSID,UML_NARWHAL_R5_B24,,,,5/21/22,n,Diseased
4,042024,1_16,BEL,CBC,4/25/24,CBC30N,1,,16,SSID,⋯,Diseased_Tissue,"50%TL sub acute, 17% acute",042024_BEL_CBC_T1_938_SSID,UML_NARWHAL_R5_B24,UML_NARWHAL_R2_B11,,,5/21/22,n,Diseased
5,042024,1_17,BEL,CBC,4/25/24,CBC30N,1,,17,SSID,⋯,Diseased_Margin,"60% TL , 20% DC",042024_BEL_CBC_T1_941_SSID,UML_NARWHAL_R5_B24,,,,5/21/22,n,"Diseased, CLP"
6,042024,1_17,BEL,CBC,4/25/24,CBC30N,1,,17,SSID,⋯,Diseased_Tissue,"60% TL , 20% DC",042024_BEL_CBC_T1_940_SSID,UML_NARWHAL_R5_B24,UML_NARWHAL_R2_B11,,,5/21/22,n,"Diseased, CLP"


[1] 254

In [56]:
nrow(metagenomics_PCR)

#check to see why there is a discrepancy from immune_sample_DNA to_metagenomics_PCR
unmatched=anti_join(meta,metagenomics_PCR, by='Tubelabel_species')
nrow(unmatched)
unique(unmatched$Tubelabel_species)

#ssids were not extracted

[1] 558

[1] 135

[1] "042024_BEL_CBC_T1_937_SSID"  "042024_BEL_CBC_T1_936_SSID" 
  [3] "042024_BEL_CBC_T1_939_SSID"  "042024_BEL_CBC_T1_941_SSID" 
  [5] "042024_BEL_CBC_T1_934_SSID"  "042024_BEL_CBC_T1_935_SSID" 
  [7] "042024_BEL_CBC_T2_1015_OANN" "042024_BEL_CBC_T2_1031_SSID"
  [9] "042024_BEL_CBC_T2_1023_MCAV" "042024_BEL_CBC_T2_1039_SSID"
 [11] "042024_BEL_CBC_T2_1036_SSID" "042024_BEL_CBC_T2_1024_OFAV"
 [13] "042024_BEL_CBC_T3_952_PAST"  "042024_BEL_CBC_T3_1058_SSID"
 [15] "042024_BEL_CBC_T3_1013_SSID" "042024_BEL_CBC_T3_963_SSID" 
 [17] "042024_BEL_CBC_T3_964_SSID"  "042024_BEL_CBC_T4_1043_OFAV"
 [19] "052022_BEL_CBC_T1_3_SSID"    "052022_BEL_CBC_T1_6_SSID"   
 [21] "052022_BEL_CBC_T1_75_SSID"   "052022_BEL_CBC_T1_36_SSID"  
 [23] "052022_BEL_CBC_T1_38_SSID"   "052022_BEL_CBC_T1_76_SSID"  
 [25] "052022_BEL_CBC_T1_18_SSID"   "052022_BEL_CBC_T1_16_SSID"  
 [27] "052022_BEL_CBC_T1_59_SSID"   "052022_BEL_CBC_T1_9_SSID"   
 [29] "052022_BEL_CBC_T1_8_SSID"    "052022_BEL_CBC_T1_15_SSID"  
 [31] "052022_BEL_CBC_T1_14_SSID"   "052022_BEL_CBC_T2_27_SSID"  
 [33] "052022_BEL_CBC_T2_36_SSID"   "052022_BEL_CBC_T2_47_SSID"  
 [35] "052022_BEL_CBC_T2_51_SSID"   "052022_BEL_CBC_T2_55_SSID"  
 [37] "052022_BEL_CBC_T2_48_SSID"   "052022_BEL_CBC_T2_49_SSID"  
 [39] "052022_BEL_CBC_T2_44_SSID"   "052022_BEL_CBC_T2_43_SSID"  
 [41] "052022_BEL_CBC_T2_16_SSID"   "052022_BEL_CBC_T2_15_SSID"  
 [43] "052022_BEL_CBC_T3_53_SSID"   "052022_BEL_CBC_T3_52_SSID"  
 [45] "052022_BEL_CBC_T3_61_SSID"   "052022_BEL_CBC_T3_62_SSID"  
 [47] "052022_BEL_CBC_T3_63_SSID"   "052022_BEL_CBC_T3_46_SSID"  
 [49] "052022_BEL_CBC_T3_64_SSID"   "052022_BEL_CBC_T3_47_SSID"  
 [51] "052022_BEL_CBC_T3_69_SSID"   "052022_BEL_CBC_T3_30_SSID"  
 [53] "052022_BEL_CBC_T3_68_SSID"   "052022_BEL_CBC_T3_67_SSID"  
 [55] "062019_BEL_CBC_T1_2_SSID"    "062019_BEL_CBC_T1_19_SSID"  
 [57] "062019_BEL_CBC_T1_11_SSID"   "062019_BEL_CBC_T1_15_SSID"  
 [59] "062019_BEL_CBC_T1_13_SSID"   "062019_BEL_CBC_T1_23_SSID"  
 [61] "062019_BEL_CBC_T1_7_SSID"    "062019_BEL_CBC_T2_27_SSID"  
 [63] "062019_BEL_CBC_T2_24_SSID"   "062019_BEL_CBC_T2_19_SSID"  
 [65] "062019_BEL_CBC_T2_3_SSID"    "062019_BEL_CBC_T2_11_SSID"  
 [67] "062019_BEL_CBC_T2_1_SSID"    "062019_BEL_CBC_T3_13_SSID"  
 [69] "062019_BEL_CBC_T3_17_SSID"   "062019_BEL_CBC_T3_18_SSID"  
 [71] "062019_BEL_CBC_T3_19_SSID"   "062019_BEL_CBC_T3_2_SSID"   
 [73] "062019_BEL_CBC_T3_29_SSID"   "062019_BEL_CBC_T3_30_SSID"  
 [75] "092023_BEL_CBC_T1_185_SSID"  "092023_BEL_CBC_T1_193_SSID" 
 [77] "092023_BEL_CBC_T1_195_SSID"  "092023_BEL_CBC_T1_197_SSID" 
 [79] "092023_BEL_CBC_T1_176_PAST"  "092023_BEL_CBC_T1_191_OANN" 
 [81] "092023_BEL_CBC_T1_189_SSID"  "092023_BEL_CBC_T1_192_SSID" 
 [83] "092023_BEL_CBC_T2_193_OANN"  "092023_BEL_CBC_T2_195_SSID" 
 [85] "092023_BEL_CBC_T2_198_PAST"  "092023_BEL_CBC_T2_220_MCAV" 
 [87] "092023_BEL_CBC_T2_189_SSID"  "092023_BEL_CBC_T2_190_SSID" 
 [89] "092023_BEL_CBC_T2_187_SSID"  "092023_BEL_CBC_T2_188_SSID" 
 [91] "092023_BEL_CBC_T2_191_OFAV"  "092023_BEL_CBC_T3_187_PAST" 
 [93] "092023_BEL_CBC_T3_199_SSID"  "092023_BEL_CBC_T3_265_SSID" 
 [95] "092023_BEL_CBC_T3_197_SSID"  "092023_BEL_CBC_T3_190_SSID" 
 [97] "092023_BEL_CBC_T3_267_SSID"  "092023_BEL_CBC_T4_89_OFAV"  
 [99] "092023_BEL_CBC_T4_85_OANN"   "122022_BEL_CBC_T1_149_SSID" 
[101] "122022_BEL_CBC_T1_148_SSID"  "122022_BEL_CBC_T1_121_SSID" 
[103] "122022_BEL_CBC_T1_147_SSID"  "122022_BEL_CBC_T1_135_SSID" 
[105] "122022_BEL_CBC_T1_142_SSID"  "122022_BEL_CBC_T1_131_SSID" 
[107] "122022_BEL_CBC_T1_125_SSID"  "122022_BEL_CBC_T1_137_SSID" 
[109] "122022_BEL_CBC_T1_138_SSID"  "122022_BEL_CBC_T1_146_SSID" 
[111] "122022_BEL_CBC_T1_145_SSID"  "122022_BEL_CBC_T2_106_SSID" 
[113] "122022_BEL_CBC_T2_104_SSID"  "122022_BEL_CBC_T2_110_SSID" 
[115] "122022_BEL_CBC_T2_111_SSID"  "122022_BEL_CBC_T2_109_SSID" 
[117] "122022_BEL_CBC_T2_108_SSID"  "122022_BEL_CBC_T2_112_SSID" 
[119] "122022_BEL_CBC_T2_113_SSID"  "122022_BEL_CBC_T2_103_SSID" 
[121] "122022_BEL_CBC_T2_105_SSID"  "12202

In [57]:
# inner join 
nonimmune_PCR <- merge(x=metagenomics_PCR,y=meta, 
                by="Tubelabel_species")
head(nonimmune_PCR)

,Tubelabel_species,Date_Extracted,Raw_ng_ul,Date_Enriched,Microbe_Location,Month_year,colony,Country,Location,CollectionDate,⋯,SampleNum,Health_status,Sampling_notes,Sample_physical_location,Extraction_physical_location,Date_sequenced,Notes,Date_DocumentedDisease,immune_y.n,Condition
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,⋯,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
1,042024_BEL_CBC_T1_1146_OANN,9_9_2024,14.6,,,042024,1_22,BEL,CBC,4/30/24,⋯,1146,Diseased_Tissue,"100% CLP; damsel predation, 30% old mortality",UML_NARWHAL_R5_B25,UML_NARWHAL_R2_B11,,,12/2/22,n,CLP
2,042024_BEL_CBC_T1_933_SSID,9_9_2024,25.4,,,042024,1_5,BEL,CBC,4/25/24,⋯,933,Diseased_Tissue,"90% CTL/ old mort , 5% DC- lots of webbing and sloughing",UML_NARWHAL_R5_B24,UML_NARWHAL_R2_B11,,,5/21/22,n,Diseased
3,042024_BEL_CBC_T1_938_SSID,9_9_2024,4.08,,,042024,1_16,BEL,CBC,4/25/24,⋯,938,Diseased_Tissue,"50%TL sub acute, 17% acute",UML_NARWHAL_R5_B24,UML_NARWHAL_R2_B11,,,5/21/22,n,Diseased
4,042024_BEL_CBC_T1_940_SSID,9_9_2024,63.2,,,042024,1_17,BEL,CBC,4/25/24,⋯,940,Diseased_Tissue,"60% TL , 20% DC",UML_NARWHAL_R5_B24,UML_NARWHAL_R2_B11,,,5/21/22,n,"Diseased, CLP"
5,042024_BEL_CBC_T2_1030_SSID,9_9_2024,24.6,,,042024,2_52,BEL,CBC,4/26/24,⋯,1030,Diseased_Tissue,80% CTL,UML_NARWHAL_R5_B24,UML_NARWHAL_R2_B11,,,5/22/22,n,Diseased
6,042024_BEL_CBC_T3_1011_SSID,9_9_2024,7.42,,,042024,3_19,BEL,CBC,4/25/24,⋯,1011,Diseased_Tissue,"95% old mort, 95% DC, webbing",UML_NARWHAL_R5_B24,UML_NARWHAL_R2_B11,,,5/20/22,n,Diseased


In [58]:
## columns I dont need in immune_meta_pcr
nonimmune_PCR[ ,c('Country', 'Location', 'Transect', 'TransectNum', 'OldTagNum', 'NewTagNum', 'Time_sampled', 'Time_processed', 'Sample_type', 'SampleNum', 'Sampling_notes', 'Date_sequenced', 'Notes', 'Date_DocumentedDisease')] <- list(NULL)

In [59]:
length(unique(nonimmune_PCR$colony))
nrow(nonimmune_PCR)

[1] 51

[1] 119

In [60]:
#make spreadsheet with all immune metagenomics 
write.csv(nonimmune_PCR, file="~/Documents/SCTLD/Coral_Code/NSF_Rapid/CBC_PCR_Plan/2_nonimmune_PCR.csv")